In [39]:
using LowLevelFEM

gmsh.initialize()

println("Running LowLevelFEM regression tests")

# ------------------------------------------------------------
# mesh
# ------------------------------------------------------------

structured_rect_mesh(n=4)

# ------------------------------------------------------------
# 1. Poisson
# ------------------------------------------------------------

println("Poisson test")

Pu = Problem([Material("surface")], type=:ScalarField, dim=2, field=:p, rhs_field=:q)

K = ∫(Grad(Pu) ⋅ Grad(Pu); Ω="surface")

ld = LoadCondition("right", q=0)
f = loadVector(Pu, [])
F = SystemVector([f])

u = solveField(K, f)

@assert length(u.a) != 1

println("✓ Poisson OK")

# ------------------------------------------------------------
# 2. Linear elasticity
# ------------------------------------------------------------

println("Elasticity test")

Pu = Problem([Material("surface")], type=:VectorField, dim=2, field=:u, rhs_field=:f)

C = [1.0 0 0;
    0 1.0 0;
    0 0 0.5]

K = ∫(ε(Pu) ⋅ C ⋅ ε(Pu); Ω="surface")

f = loadVector(Pu, [])
#F = SystemVector(f)

u = solveField(K, f)

@assert length(u.a) != 1

println("✓ Elasticity OK")

# ------------------------------------------------------------
# 3. Stokes (multifield)
# ------------------------------------------------------------

println("Stokes test")

Pu = Problem([Material("surface")], type=:VectorField, dim=2, field=:v)
Pp = Problem([Material("surface")], type=:ScalarField, dim=2, field=:p)

A = ∫(ε(Pu) ⋅ ε(Pu); Ω="surface")
B = ∫(Div(Pu) ⋅ Pp; Ω="surface")
C = ∫(Pp ⋅ Div(Pu); Ω="surface")

K = SystemMatrix([A B'; B C])

fu = loadVector(Pu, [])
fp = loadVector(Pp, [])

F = SystemVector([fu, fp])

bcv1 = BoundaryCondition("top", problem=Pu, vx=0, vy=0)
bcv2 = BoundaryCondition("bottom", problem=Pu, vx=0, vy=0)
bcp = BoundaryCondition("leftbottom", problem=Pp, p=0)

(u, p) = solveField(K, F, support=[bcv1, bcv2, bcp])

@assert u.model === Pu
@assert p.model === Pp

println("✓ Stokes OK")

println("All regression tests passed")

Running LowLevelFEM regression tests
Poisson test
✓ Poisson OK
Elasticity test
✓ Elasticity OK
Stokes test
✓ Stokes OK
All regression tests passed


In [40]:
gmsh.finalize()